# 06 — Proponentes por Proceso SECOP II

**Fuente**: `proponentesPorProceso` (Dataset ID: `hgi6-6wh3`)  
**Granularidad**: 1 fila = 1 proponente que presentó oferta en 1 proceso  
**Columnas disponibles**: 9  

**Objetivo**: Descargar los proponentes que participaron en procesos de contratación de Obra Pública,
para luego extraer su información financiera del RUP mediante el scraper automático.

**Estrategia de descarga**: El dataset completo tiene ~2.25M registros de todos los tipos de contrato.
Filtramos desde la API usando los `id_del_proceso` de procesos de Obra que ya tenemos en parquet.

**Llave de integración**: `id_procedimiento` (proponentes) = `id_del_proceso` (procesosDeContratacion)

| Campo API | Tipo | Descripción |
|---|---|---|
| id_procedimiento | Texto | Identificador del procedimiento (CO1.REQ.XXX) |
| fecha_publicaci_n | Timestamp | Fecha de publicación del proceso |
| nombre_procedimiento | Texto | Nombre del proceso |
| nit_entidad | Número | NIT de la entidad contratante |
| codigo_entidad | Número | Código de la entidad contratante |
| entidad_compradora | Texto | Nombre de la entidad compradora |
| proveedor | Texto | Nombre del proveedor/proponente |
| nit_proveedor | Texto | NIT del proveedor |
| codigo_proveedor | Número | Código SECOP II del proveedor |

---

## 1. Configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sodapy import Socrata
import warnings
import time

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Constantes de conexión
APP_TOKEN  = 'nrnMDVHrooMY3wZIGrYfPsAq3'
DOMAIN     = 'www.datos.gov.co'
DATASET_ID = 'hgi6-6wh3'  # proponentesPorProceso
DATA_DIR   = '../data/'

client = Socrata(DOMAIN, APP_TOKEN, timeout=120)

def fetch_with_retry(client, dataset_id, max_retries=5, **kwargs):
    """Consulta Socrata con reintentos y backoff."""
    for attempt in range(max_retries):
        try:
            return client.get(dataset_id, **kwargs)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 10 * (attempt + 1)
                print(f'    Error: {type(e).__name__}. Reintentando en {wait}s... (intento {attempt+2}/{max_retries})')
                time.sleep(wait)
            else:
                raise

print("OK")

OK


## 2. Cargar IDs de procesos de Obra

Cargamos el parquet de `procesosDeContratacion` filtrado por Obra y extraemos los `id_del_proceso` únicos (formato `CO1.REQ.XXX`).

In [2]:
df_procesos = pd.read_parquet(DATA_DIR + 'procesos_contratacion_obra.parquet')
obra_ids = df_procesos['id_del_proceso'].dropna().unique().tolist()

print(f"Registros en parquet:       {len(df_procesos):,}")
print(f"id_del_proceso únicos:      {len(obra_ids):,}")
print(f"Formato ejemplo:            {obra_ids[0]}")

Registros en parquet:       138,112
id_del_proceso únicos:      112,891
Formato ejemplo:            CO1.REQ.848621


## 3. Verificar nombres de campos en la API

Descargamos 1 registro para confirmar los nombres reales de los campos (Socrata los normaliza).

In [3]:
sample = fetch_with_retry(client, DATASET_ID, limit=1)
api_columns = list(sample[0].keys())
print("Campos reales en la API:")
for col in api_columns:
    print(f"  - {col}: {sample[0][col][:80] if isinstance(sample[0][col], str) else sample[0][col]}")

Campos reales en la API:
  - id_procedimiento: CO1.REQ.10159505
  - fecha_publicaci_n: 2026-02-11T00:00:00.000
  - nombre_procedimiento: SUMINISTRO DE PAÑALES DESECHABLES Y PAÑITOS HuMEDOS ESM-BAS06
  - nit_entidad: 901541245
  - codigo_entidad: 702962226
  - entidad_compradora: ESM BATALLoN ASPC NO. 6
  - proveedor: INTERCOMERCIAL MEDICA
  - nit_proveedor: 830501223
  - codigo_proveedor: 700150014


## 4. Descarga por lotes

Consultamos `proponentesPorProceso` filtrando con `WHERE id_procedimiento IN (...)` en lotes de 150 IDs.
Mismo patrón que el notebook 03 (adiciones).

In [4]:
chunk_size = 150
all_records = []
total_chunks = (len(obra_ids) + chunk_size - 1) // chunk_size

print(f"IDs de procesos de Obra:  {len(obra_ids):,}")
print(f"Lotes de {chunk_size} IDs:          {total_chunks}")
print()

for i in range(0, len(obra_ids), chunk_size):
    chunk = obra_ids[i:i+chunk_size]
    chunk_num = i // chunk_size + 1

    # Construir cláusula WHERE IN
    ids_str = ", ".join([f"'{pid}'" for pid in chunk])
    where_clause = f"id_procedimiento IN ({ids_str})"

    # Descargar todos los proponentes para este lote (con paginación)
    offset = 0
    while True:
        batch = fetch_with_retry(client, DATASET_ID,
                                 where=where_clause,
                                 limit=50000,
                                 offset=offset)
        if not batch:
            break
        all_records.extend(batch)
        if len(batch) < 50000:
            break
        offset += 50000

    if chunk_num % 50 == 0 or chunk_num == total_chunks:
        print(f"  Lote {chunk_num}/{total_chunks} — Registros acumulados: {len(all_records):,}")

print(f"\n✓ Descarga completa: {len(all_records):,} registros")

IDs de procesos de Obra:  112,891
Lotes de 150 IDs:          753



  Lote 50/753 — Registros acumulados: 28,311


  Lote 100/753 — Registros acumulados: 49,046


  Lote 150/753 — Registros acumulados: 68,958


  Lote 200/753 — Registros acumulados: 88,334


...

  Lote 600/753 — Registros acumulados: 226,037


  Lote 650/753 — Registros acumulados: 242,518


  Lote 700/753 — Registros acumulados: 258,438


  Lote 750/753 — Registros acumulados: 275,628


  Lote 753/753 — Registros acumulados: 276,770

✓ Descarga completa: 276,770 registros


In [5]:
df = pd.DataFrame.from_records(all_records)
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

Shape: (276770, 9)
Columnas: ['id_procedimiento', 'fecha_publicaci_n', 'nombre_procedimiento', 'nit_entidad', 'codigo_entidad', 'entidad_compradora', 'proveedor', 'nit_proveedor', 'codigo_proveedor']


,id_procedimiento,fecha_publicaci_n,nombre_procedimiento,nit_entidad,codigo_entidad,entidad_compradora,proveedor,nit_proveedor,codigo_proveedor
0,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSORCIO SAIFOR,1102868937,734498561
1,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSORCIO VENTURA & CO.,No Definido,734861826
2,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,CONSTRUCTORA AHA S.A.S,830010208,701699522
3,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,SAMAR CONSTRUCTORES Y CONSULTORES SAS,900088297,700844095
4,CO1.REQ.10064540,2026-01-30T00:00:00.000,CONTRATAR LA TERMINACIoN MEJORAMIENTO REMODELACIoN ADECUACIoN MODIFICACIoN MANTENIMIENTO PREVENT...,800141686,701746083,ARMADA NACIONAL - BASE NAVAL NO. 3 ARC LEGUIZAMO.*,MARCELINO,No Definido,734906878


## 5. Perfilado rápido

In [6]:
print("=== Perfilado básico ===\n")
print(f"Registros totales:          {len(df):,}")
print(f"Procesos únicos:            {df['id_procedimiento'].nunique():,}")
print(f"Proponentes únicos (NIT):   {df['nit_proveedor'].nunique():,}")
print(f"Entidades únicas:           {df['entidad_compradora'].nunique():,}")

print(f"\n--- Nulos por columna ---")
print(df.isnull().sum().to_string())

print(f"\n--- Valores 'No Definido' ---")
for col in df.columns:
    nd = (df[col] == 'No Definido').sum()
    if nd > 0:
        print(f"  {col}: {nd:,}")

print(f"\n--- Proponentes por proceso ---")
props_por_proc = df.groupby('id_procedimiento')['nit_proveedor'].count()
print(props_por_proc.describe())

=== Perfilado básico ===

Registros totales:          276,770
Procesos únicos:            33,171
Proponentes únicos (NIT):   23,759
Entidades únicas:           2,264

--- Nulos por columna ---
id_procedimiento        0
fecha_publicaci_n       3
nombre_procedimiento    0
nit_entidad             0
codigo_entidad          0
entidad_compradora      0
proveedor               0
nit_proveedor           0
codigo_proveedor        0

--- Valores 'No Definido' ---
  nit_proveedor: 93,103

--- Proponentes por proceso ---
count    33171.000000
mean         8.343734
std         15.770741
min          1.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        255.000000
Name: nit_proveedor, dtype: float64


## 6. Cobertura vs procesosDeContratacion

¿Cuántos de los 112K procesos de Obra tienen al menos 1 proponente en este dataset?

In [7]:
ids_con_proponentes = set(df['id_procedimiento'].unique())
ids_obra = set(obra_ids)

match = ids_con_proponentes & ids_obra
sin_proponentes = ids_obra - ids_con_proponentes

print(f"Procesos de Obra (parquet):     {len(ids_obra):,}")
print(f"Procesos con proponentes:       {len(match):,}  ({len(match)/len(ids_obra)*100:.1f}%)")
print(f"Procesos sin proponentes:       {len(sin_proponentes):,}  ({len(sin_proponentes)/len(ids_obra)*100:.1f}%)")

Procesos de Obra (parquet):     112,891


Procesos con proponentes:       33,171  (29.4%)
Procesos sin proponentes:       79,720  (70.6%)


## 7. NITs únicos de proponentes

Estos son los NITs que se pueden enviar al scraper RUP para obtener información financiera.

In [8]:
nits_validos = df[
    (df['nit_proveedor'].notna()) &
    (df['nit_proveedor'] != 'No Definido') &
    (df['nit_proveedor'] != '')
]['nit_proveedor'].unique()

print(f"NITs de proponentes únicos:     {len(nits_validos):,}")
print(f"Primeros 10: {list(nits_validos[:10])}")

NITs de proponentes únicos:     23,758
Primeros 10: ['1102868937', '830010208', '900088297', '900392539', '901058985', '31655741', '0001013594558', '901346178', '901590857', '901455911']


## 8. Dataset unificado de proveedores para scraper RUP

Combinamos NITs de ambas fuentes en un solo dataset a nivel NIT:
- **procesos**: proponentes de `proponentesPorProceso` (Obra)
- **electronicos**: proveedores adjudicados de `contratosElectronicos` (Obra)

Columnas RUP vacías que llenará el scraper.

In [ ]:
import re

# --- NITs de proponentes (procesos) ---
raw_prop = df['nit_proveedor'].copy()
raw_prop = raw_prop.apply(lambda x: x.split('-')[0] if isinstance(x, str) and re.match(r'^\d+-\d$', x) else x)
mask_prop = raw_prop.notna() & (raw_prop != 'No Definido') & raw_prop.str.match(r'^\d{6,10}$', na=False)

df_prop = df[mask_prop][['proveedor']].copy()
df_prop['nit'] = raw_prop[mask_prop]
nits_proc = df_prop.groupby('nit')['proveedor'].first().reset_index()
nits_proc['origen'] = 'procesos'
nits_proc.rename(columns={'proveedor': 'nombre_proveedor'}, inplace=True)
print(f"NITs limpios de procesos: {len(nits_proc):,}")

# --- NITs de contratos electrónicos ---
df_ce = pd.read_parquet(DATA_DIR + 'contratos_electronicos_obra.parquet')
raw_ce = df_ce['documento_proveedor'].copy()
raw_ce = raw_ce.apply(lambda x: x.split('-')[0] if isinstance(x, str) and re.match(r'^\d+-\d$', x) else x)
mask_ce = raw_ce.notna() & (raw_ce != 'No Definido') & raw_ce.str.match(r'^\d{6,10}$', na=False)

df_ce_nits = df_ce[mask_ce][['proveedor_adjudicado']].copy()
df_ce_nits['nit'] = raw_ce[mask_ce]
nits_elec = df_ce_nits.groupby('nit')['proveedor_adjudicado'].first().reset_index()
nits_elec['origen'] = 'electronicos'
nits_elec.rename(columns={'proveedor_adjudicado': 'nombre_proveedor'}, inplace=True)
print(f"NITs limpios de electrónicos: {len(nits_elec):,}")

# --- Unificar a nivel NIT ---
df_all = pd.concat([nits_proc, nits_elec], ignore_index=True)

origen_map = df_all.groupby('nit')['origen'].apply(
    lambda x: 'ambos' if set(x) == {'procesos', 'electronicos'} else x.iloc[0]
)
nombre_map = df_all.groupby('nit')['nombre_proveedor'].first()

df_unificado = pd.DataFrame({
    'nit': origen_map.index,
    'nombre_proveedor': nombre_map.values,
    'origen': origen_map.values,
})

print(f"\n=== Dataset unificado ===")
print(f"Total NITs únicos:    {len(df_unificado):,}")
print(f"  procesos:           {(df_unificado['origen'] == 'procesos').sum():,}")
print(f"  electronicos:       {(df_unificado['origen'] == 'electronicos').sum():,}")
print(f"  ambos:              {(df_unificado['origen'] == 'ambos').sum():,}")

In [ ]:
# Agregar columnas RUP vacías (las llenará el scraper)
rup_cols = [
    'rup_estado',           # estado_matricula
    'rup_tamano',           # codigo_tamano_empresa
    'rup_empleados',        # numero_empleados
    'rup_municipio',        # municipio_comercial
    'rup_sanciones',        # count de sanciones
    'rup_multas',           # count de multas
    'rup_inhabilidad',      # inhabilidad
    'rup_ano_financiero',   # ano_informacion_financiera
    'rup_activo_total',     # activo_total
    'rup_patrimonio',       # patrimonio_neto
    'rup_ingresos',         # ingresos_actividad_ordinaria
    'rup_utilidad_neta',    # resultado_del_periodo
    'rup_idx_liquidez',     # AC / PC
    'rup_idx_endeudamiento',# PT / AT
    'rup_idx_cobertura',    # UO / GI
    'rup_rent_patrimonio',  # UN / PN
    'rup_rent_activo',      # UN / AT
]

for col in rup_cols:
    df_unificado[col] = np.nan

# Columnas de control
df_unificado['procesado'] = False

# procesar = False para consorcios/UT y NITs todo ceros (no tienen RUP)
nombre_upper = df_unificado['nombre_proveedor'].str.upper().fillna('')
todo_ceros = df_unificado['nit'].str.match(r'^0+$')
consorcios = nombre_upper.str.contains(r'CONSORCIO|UNION TEMPORAL|UNIoN TEMPORAL|U\.T\.|UT ', regex=True)
df_unificado['procesar'] = ~(consorcios | todo_ceros)

# Ordenar
df_unificado = df_unificado.sort_values('nit').reset_index(drop=True)

print(f"Columnas finales ({len(df_unificado.columns)}):")
for c in df_unificado.columns:
    print(f"  {c}")

print(f"\nShape: {df_unificado.shape}")
print(f"procesar=True:  {df_unificado['procesar'].sum():,}")
print(f"procesar=False: {(~df_unificado['procesar']).sum():,}")
df_unificado.head()

## 9. Guardar

In [ ]:
# Parquet crudo de proponentes (descarga original)
output_raw = DATA_DIR + 'proponentes_por_proceso_obra.parquet'
df.to_parquet(output_raw, index=False)
print(f"Guardado: {output_raw}")
print(f"  {len(df):,} registros, {df.shape[1]} columnas")

# Dataset unificado para scraper RUP
output_rup = DATA_DIR + 'proveedores_rup.parquet'
df_unificado.to_parquet(output_rup, index=False)
print(f"\nGuardado: {output_rup}")
print(f"  {len(df_unificado):,} NITs únicos, {df_unificado.shape[1]} columnas")
print(f"  Pendientes de procesar: {(~df_unificado['procesado']).sum():,}")